<a href="https://colab.research.google.com/github/Decoding-Data-Science/CommunityWorkshops/blob/main/bootcampoct26/notebooks/Day2_RAG_and_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DAY 2 | Give DecoBot the Policy: RAG and Chunking

**Decoding Data Science | AI Accelerator Bootcamp (CPD UK)**

| | |
|---|---|
| **Time** | about 2 hours |
| **Goal** | Let the AI answer from a real PDF, and learn why *how you cut the document* decides answer quality |
| **You will build** | A policy search engine over the DDS Leave Policy, tested with three chunking strategies |
| **Files needed** | `DDS_Leave_Policy_v2.pdf` (you upload it in Part 0) |
| **Tools** | Google Colab + Gemini + LlamaIndex |

---

### Where we are

```
Day 1   System prompt + clarity   (done)  DecoBot behaves well but only knows 6 pasted facts
Day 2   RAG over the PDF          (today) DecoBot searches the real policy
Day 3   SQL + tools + agent       (next)  DecoBot checks each person's balance
```

### What is RAG? (the 4 steps)

```
 PDF  --1. split-->  chunks  --2. embed-->  numbers  --3. search-->  best chunks  --4. AI answers using them
```

Retrieval-Augmented Generation = **find** the right paragraphs first, **then** let the AI answer from them.

## Part 0 | Setup (10 min)

In [1]:
%pip install -q llama-index-core llama-index-llms-google-genai llama-index-embeddings-google-genai pypdf

import importlib.metadata as md
for pkg in ["llama-index-core", "llama-index-llms-google-genai", "llama-index-embeddings-google-genai", "pypdf"]:
    print(f"{pkg:40s}", md.version(pkg))

llama-index-core                         0.14.25
llama-index-llms-google-genai            0.11.3
llama-index-embeddings-google-genai      0.6.0
pypdf                                    6.21.4


Upload the policy PDF (`DDS_Leave_Policy_v2.pdf`) when the file picker opens.

In [2]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving DDS_Leave_Policy_v2.pdf to DDS_Leave_Policy_v2.pdf
Uploaded: ['DDS_Leave_Policy_v2.pdf']


> **Before you run this cell:** click the key icon on the left of Colab (Secrets), add a secret named `GOOGLE_API_KEY`
> with your Gemini API key, and switch on "Notebook access". Never paste your key into a cell.

In [4]:
import os
from google.colab import userdata

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("gemini2")
except Exception as error:
    raise RuntimeError(
        "Colab secret GOOGLE_API_KEY not found. Click the key icon on the left of Colab, "
        "add a secret named GOOGLE_API_KEY with your Gemini API key, switch on Notebook access, "
        "then run this cell again."
    ) from error

print("API key loaded:", bool(os.environ["GOOGLE_API_KEY"]))

API key loaded: True


In [6]:
import nest_asyncio
nest_asyncio.apply()

from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding

MODEL = "gemini-3.8-flash"                # change only these two lines if Google renames a model
EMBED_MODEL = "gemini-embedding-001"

Settings.llm = GoogleGenAI(model=MODEL, temperature=0.1)
Settings.embed_model = GoogleGenAIEmbedding(model_name=EMBED_MODEL, embed_batch_size=50)

print(Settings.llm.complete("Say hello to the DDS team in one short sentence.").text)

Hello to the wonderful DDS team!


---
## Part A | What is an embedding? (10 min)

An **embedding** turns a sentence into a list of numbers that captures its *meaning*. Sentences with similar meaning get similar numbers.
That is how a computer can find *"vacation days"* when the policy says *"annual leave"*.

In [7]:
import numpy as np

embed = Settings.embed_model
vec = embed.get_text_embedding("annual leave")
print("One sentence became", len(vec), "numbers. First five:", [round(x, 3) for x in vec[:5]])

def similarity(a, b):
    va, vb = np.array(embed.get_text_embedding(a)), np.array(embed.get_text_embedding(b))
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))

query = "How many vacation days do I get?"
for text in ["Annual leave: 22 working days per year",
             "Medical certificate is required for sick leave",
             "The office cafeteria opens at 8 am"]:
    print(f"{similarity(query, text):.3f}  <-  {text}")

One sentence became 3072 numbers. First five: [-0.032, 0.014, -0.013, -0.07, 0.002]
0.897  <-  Annual leave: 22 working days per year
0.820  <-  Medical certificate is required for sick leave
0.763  <-  The office cafeteria opens at 8 am


**Checkpoint.** The query says *vacation* but the best match says *annual leave*. No keyword is shared, yet the meaning matched.
That is why RAG uses embeddings instead of Ctrl+F.

---
## Part B | Load the PDF and build a first index (15 min)

In [8]:
from pypdf import PdfReader
from llama_index.core import Document

pdf = PdfReader("DDS_Leave_Policy_v2.pdf")
documents = [Document(text=page.extract_text(), metadata={"page": i + 1}) for i, page in enumerate(pdf.pages)]
print("Pages loaded:", len(documents))
print("\nFirst 400 characters of page 1:\n")
print(documents[0].text[:400])

Pages loaded: 4

First 400 characters of page 1:

Leave Policy - Decoding Data Science
Document ID: DDS-HR-POL-001 | Version 2.0 | Effective date: 1 July 2026
Internal - Synthetic demo document for AI training purposes. Not legal advice.
This policy defines all leave entitlements at DDS, how leave accrues, how to request it, and how requests are
approved. It replaces Leave Policy v1 (March 2026) in full. Where UAE Federal Decree-Law No. 33 of 202


In [9]:
from llama_index.core import VectorStoreIndex

# Default settings: LlamaIndex chooses the chunk size for us
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=3)

response = query_engine.query("How much notice do I need to book 12 working days of annual leave?")
print(response)

You need to provide 30 calendar days' notice to book 12 working days of annual leave.


**Seeing is believing.** Every answer comes from *retrieved chunks*. Let us look at exactly which chunks were used and how well they scored.

In [10]:
def show_sources(response, chars=260):
    for i, n in enumerate(response.source_nodes, 1):
        text = " ".join(n.node.get_content().split())
        print(f"[{i}] score={n.score:.3f} | {text[:chars]}...\n")

show_sources(response)

[1] score=0.680 | Leave Policy - Decoding Data Science Document ID: DDS-HR-POL-001 | Version 2.0 | Effective date: 1 July 2026 Internal - Synthetic demo document for AI training purposes. Not legal advice. This policy defines all leave entitlements at DDS, how leave accrues, ho...

[2] score=0.670 | 3.5 Peak periods During flagship delivery windows (major bootcamps, the annual conference season, and product launches), approvals may be restricted. HR publishes restricted weeks at the start of each quarter. Requests in restricted weeks require Head-of-Depar...

[3] score=0.655 |  An additional period of up to 45 calendar days unpaid may be taken for medical reasons related to the pregnancy or delivery, with a medical certificate. 7.2 Nursing breaks For 6 months following return from maternity leave, the employee is entitled to one pa...



> **Lesson 1.** When a RAG answer is wrong, the model is rarely the problem. Look at the retrieved chunks first.
> Wrong chunks in, wrong answer out.

---
## Part C | The chunking lab (45 min)

Before search, the document is cut into **chunks**. How you cut decides what the search can find.

```
Too big     ->  a chunk mixes several topics, the search gets noisy
Too small   ->  a rule is cut in half, the answer loses its context
Cut by meaning (sections) -> each chunk is one complete idea
```

We will compare **three strategies** on the same policy:

| Strategy | How it cuts | Intuition |
|---|---|---|
| **A. Big chunks** | about 1000 tokens, no overlap | almost one chunk per page |
| **B. Small chunks** | about 120 tokens, small overlap | many tiny pieces |
| **C. By section** | at each numbered heading (1., 2., 3. ...) | one chunk per policy section, with the section title attached |

In [11]:
import re
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode

full_text = "\n".join(d.text for d in documents).replace("\x7f", "-")   # clean PDF bullet characters

# Strategy A: big chunks
nodes_a = SentenceSplitter(chunk_size=1000, chunk_overlap=0).get_nodes_from_documents(documents)

# Strategy B: small chunks with a little overlap
nodes_b = SentenceSplitter(chunk_size=120, chunk_overlap=20).get_nodes_from_documents(documents)

# Strategy C: one chunk per numbered section, title kept as metadata
parts = re.split(r"(?m)^(?=\d{1,2}\.\s+[A-Z])", full_text)
nodes_c = []
for part in parts:
    part = part.strip()
    if not part:
        continue
    title = part.splitlines()[0]                    # e.g. "3. Annual Leave"
    nodes_c.append(TextNode(text=part, metadata={"section": title}))

for name, nodes in [("A big", nodes_a), ("B small", nodes_b), ("C by section", nodes_c)]:
    sizes = [len(n.get_content()) for n in nodes]
    print(f"{name:14s} chunks={len(nodes):3d}   avg size={sum(sizes)//len(sizes):5d} chars   biggest={max(sizes):5d}")

A big          chunks=  4   avg size= 2765 chars   biggest= 3637
B small        chunks= 28   avg size=  416 chars   biggest=  544
C by section   chunks= 13   avg size=  850 chars   biggest= 1544


### Look at what each strategy did to one important rule

The notice table (section 3.4) says: *1-2 days need 7 days notice, 3-9 days need 14, 10 or more days need 30.*
Is that table still in one piece?

In [12]:
def find_chunk(nodes, phrase):
    for n in nodes:
        if phrase in " ".join(n.get_content().split()):
            return n.get_content()
    return "(not found)"

phrase = "10 or more working days requested"
for name, nodes in [("A big", nodes_a), ("B small", nodes_b), ("C by section", nodes_c)]:
    chunk = " ".join(find_chunk(nodes, phrase).split())
    print(f"\n=== {name} ({len(chunk)} chars) ===")
    print(chunk[:600])


=== A big (3637 chars) ===
Leave Policy - Decoding Data Science Document ID: DDS-HR-POL-001 | Version 2.0 | Effective date: 1 July 2026 Internal - Synthetic demo document for AI training purposes. Not legal advice. This policy defines all leave entitlements at DDS, how leave accrues, how to request it, and how requests are approved. It replaces Leave Policy v1 (March 2026) in full. Where UAE Federal Decree-Law No. 33 of 2021 (the UAE Labour Law) provides a greater benefit, the law prevails. Note: this is a synthetic, training-friendly policy created for demos, onboarding simulations, and HR-chatbot prototypes. It is not

=== B small (271 chars) ===
3-9 working days requested: 14 calendar days' notice. 10 or more working days requested: 30 calendar days' notice. Managers respond to leave requests within 3 business days. Employees should not book non-refundable travel before approval is confirmed in the leave system.

=== C by section (1446 chars) ===
3. Annual Leave 3.1 Entitlement Ful

**Checkpoint.** Compare the three printouts.

- In which strategy is the notice rule **complete** (all three notice levels together)?
- In which strategy is it **cut off** or mixed with unrelated rules?

### Score the strategies with a mini test

We will test retrieval, not the AI's writing. For each question we know which phrase **must** appear in the retrieved chunks.
If the phrase is there, the strategy **PASSES** that question.

In [13]:
tests = [
    ("How much notice for 12 working days of annual leave?", "10 or more working days requested: 30 calendar days"),
    ("What is the maximum number of days I can carry over?", "maximum of 10 unused"),
    ("When do my carried-over days expire?",               "31 march (end of q1)"),
    ("How long is maternity leave and is it paid?",        "60 calendar days at full pay"),
    ("When do I need a medical certificate?",              "2 or more consecutive working days"),
    ("Can I take Hajj leave more than once?",              "once during their employment"),
    ("How fast must my manager reply to a request?",       "within 3 business days"),
    ("What is the longest annual leave block allowed?",    "15 consecutive working days"),
]

def hit_rate(nodes, top_k=3, verbose=True):
    retriever = VectorStoreIndex(nodes).as_retriever(similarity_top_k=top_k)
    hits = 0
    for question, phrase in tests:
        found = retriever.retrieve(question)
        text = " ".join(" ".join(n.node.get_content().split()).lower() for n in found)
        ok = phrase in text
        hits += ok
        if verbose:
            print("  PASS" if ok else "  MISS", "-", question)
    return hits / len(tests)

results = {}
for name, nodes in [("A big", nodes_a), ("B small", nodes_b), ("C by section", nodes_c)]:
    print(f"\n{name}")
    results[name] = hit_rate(nodes)

print("\nRESULT")
for name, score in results.items():
    print(f"{name:14s} {score:.0%}  " + "#" * int(score * 20))


A big
  PASS - How much notice for 12 working days of annual leave?
  PASS - What is the maximum number of days I can carry over?
  PASS - When do my carried-over days expire?
  PASS - How long is maternity leave and is it paid?
  PASS - When do I need a medical certificate?
  PASS - Can I take Hajj leave more than once?
  PASS - How fast must my manager reply to a request?
  PASS - What is the longest annual leave block allowed?

B small
  PASS - How much notice for 12 working days of annual leave?
  PASS - What is the maximum number of days I can carry over?
  PASS - When do my carried-over days expire?
  MISS - How long is maternity leave and is it paid?
  PASS - When do I need a medical certificate?
  PASS - Can I take Hajj leave more than once?
  PASS - How fast must my manager reply to a request?
  PASS - What is the longest annual leave block allowed?

C by section
  PASS - How much notice for 12 working days of annual leave?
  PASS - What is the maximum number of days I can ca

**Discuss in pairs (5 min).**

- Which strategy won? Is that what you expected?
- Which questions were missed, and what is special about them? *(Hint: look at where the answer sits in the document.)*
- Embeddings are not deterministic across models. If your result differs from your neighbour's, what does that tell you about testing?

> **Lesson 2.** There is no universally best chunk size. The best strategy depends on the document. Policies have numbered sections,
> so **cutting by section** usually keeps each rule whole (check your own numbers above). Contracts, emails and manuals need different cuts.
> Always **measure** with test questions instead of guessing.

### Experiment: how many chunks should we retrieve? (`top_k`)

Retrieving 1 chunk risks missing the answer. Retrieving 8 floods the AI with noise (and costs more). Try it on the section strategy.

In [14]:
for k in [1, 2, 3, 5]:
    print(f"top_k = {k}  ->  hit rate {hit_rate(nodes_c, top_k=k, verbose=False):.0%}")

top_k = 1  ->  hit rate 100%
top_k = 2  ->  hit rate 100%
top_k = 3  ->  hit rate 100%
top_k = 5  ->  hit rate 100%


**Your turn.** Change the chunk size of strategy B (try 60, 200, 400) and the overlap (0, 20, 50). Does the hit rate move?
Write your best combination here: ________________

In [15]:
for size, overlap in [(60, 0), (200, 30), (400, 50)]:
    nodes = SentenceSplitter(chunk_size=size, chunk_overlap=overlap).get_nodes_from_documents(documents)
    print(f"chunk_size={size:3d} overlap={overlap:2d} -> chunks={len(nodes):3d}  hit rate {hit_rate(nodes, verbose=False):.0%}")

chunk_size= 60 overlap= 0 -> chunks= 59  hit rate 100%
chunk_size=200 overlap=30 -> chunks= 17  hit rate 100%
chunk_size=400 overlap=50 -> chunks=  9  hit rate 100%


---
## Part D | The final policy engine (20 min)

We keep the by-section chunks (or your best strategy) and add three things:
1. A **prompt** that forces the AI to answer only from the policy and to name the section.
2. A clear **"I could not find that"** answer, so it never invents rules.
3. A single function, `ask_policy()`, which becomes a **tool** on Day 3.

In [16]:
from llama_index.core import PromptTemplate

index = VectorStoreIndex(nodes_c)

qa_prompt = PromptTemplate(
    "You are DecoBot, the leave assistant for Decoding Data Science (DDS).\n"
    "Answer ONLY from the policy excerpts below. Quote numbers exactly and mention the section.\n"
    "If the answer is not in the excerpts, reply exactly: "
    "'I could not find that in the DDS leave policy. Please contact People & Culture.'\n"
    "Keep the answer to 4 short lines.\n"
    "---------------------\n{context_str}\n---------------------\n"
    "Question: {query_str}\nAnswer: "
)

query_engine = index.as_query_engine(similarity_top_k=3, text_qa_template=qa_prompt)

def ask_policy(question):
    response = query_engine.query(question)
    print("QUESTION:", question)
    print("ANSWER  :", str(response).strip())
    sections = sorted({n.node.metadata.get("section", "?") for n in response.source_nodes})
    print("SOURCES :", " | ".join(sections))
    print()
    return str(response)

_ = ask_policy("How much notice do I need to book 12 working days of annual leave?")

QUESTION: How much notice do I need to book 12 working days of annual leave?
ANSWER  : According to section 3.4 (Notice periods for annual leave requests):
For 10 or more working days requested, 30 calendar days' notice is required.
Therefore, booking 12 working days requires 30 calendar days' notice.
Managers will respond to your request within 3 business days.
SOURCES : 1. Leave Entitlements at a Glance | 12. Governance | 3. Annual Leave



### Scenario cards: real questions DDS employees ask

Run each one. For every answer, check: **is it correct, did it cite a section, and did it stay inside the policy?**

In [17]:
scenarios = [
    "My grandmother passed away. How many days of leave do I get?",
    "I am expecting a baby in March. What leave am I entitled to?",
    "Can I get my unused annual leave paid out in cash?",
    "Does Eid count as one of my annual leave days?",
    "I want to go for Hajj. What is the rule?",
    "I was sick on Monday and Tuesday. What do I need to do?",
    "My manager keeps asking me to cancel my approved leave. Is that allowed?",
]
for s in scenarios:
    _ = ask_policy(s)

QUESTION: My grandmother passed away. How many days of leave do I get?
ANSWER  : According to section 8.1, you are entitled to bereavement leave.
You receive 3 working days at full pay for the death of a grandparent.
Notify your manager as soon as possible.
Documentation may be requested.
SOURCES : 1. Leave Entitlements at a Glance | 2. Definitions | 8. Other Leave Types

QUESTION: I am expecting a baby in March. What leave am I entitled to?
ANSWER  : Per Section 7.1, maternity leave is 60 calendar days at full pay, plus up to 45 calendar days unpaid for medical reasons.
Per Section 7.3, fathers receive 5 working days of paternity leave at full pay within 6 months of the child's birth.
Per Section 7.4, each parent may take an additional 5 working days of paid parental leave within 6 months of the birth.
Per Section 7.2, returning mothers receive one paid nursing break of 1 hour per day for 6 months.
SOURCES : 1. Leave Entitlements at a Glance | 7. Maternity, Paternity and Parental Leav

### The trap questions

A good bot must also know what it does **not** know.

In [18]:
traps = [
    "What is the dress code at DDS?",
    "How many annual leave days do I have left this year?",
]
for t in traps:
    _ = ask_policy(t)

QUESTION: What is the dress code at DDS?
ANSWER  : I could not find that in the DDS leave policy. Please contact People & Culture.
SOURCES : 2. Definitions | 6. Public Holidays | 7. Maternity, Paternity and Parental Leave

QUESTION: How many annual leave days do I have left this year?
ANSWER  : I could not find that in the DDS leave policy. Please contact People & Culture.
SOURCES : 1. Leave Entitlements at a Glance | 3. Annual Leave | 4. Carry-Over, Expiry and Encashment



**Checkpoint.**

- *Dress code*: the bot should say it could not find it. Did it?
- *How many days do **I** have left?*: the policy explains the **rules**, but your **balance** is personal data that lives in a **database**.
  A document cannot answer it, no matter how well we chunk it.

> **Lesson 3.** RAG answers *"what are the rules?"*. It cannot answer *"what is **my** balance?"*.
> Tomorrow we connect DecoBot to the DDS leave database with SQL, and let an **agent** decide when to use the policy and when to use the data.

---
## Wrap-up

### What we built today
```
PDF -> sections (chunks) -> embeddings -> search -> best 3 chunks -> Gemini answers with the section number
```

### The chunking checklist

| Question to ask | Why it matters |
|---|---|
| Does each chunk hold **one complete idea**? | Rules cut in half give half answers |
| Does the chunk carry its **title or section**? | Helps both search and citations |
| Did I **measure** with test questions? | Guessing chunk sizes is how RAG projects fail |
| Is `top_k` big enough but not noisy? | Too few misses, too many confuse |
| Does the bot say **"I do not know"** when it should? | Trust is built by refusing, not by guessing |

### Self-check
1. In one sentence, what does an embedding do?
2. Why did cutting by section help with the notice table?
3. Which kinds of questions can RAG over a PDF **not** answer?

### Tomorrow (Day 3)
SQL on the employee leave database, three tools, and an agent that chooses between them.